In [2]:
# 03_generation.ipynb -- generate under gap_given vs gap_blind conditions (circularity ablation)

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 03: generate (rating, rationale) under TWO conditions for the circularity ablation ---
# gap_given = generator is told the notch gap (informed analyst); gap_blind = not told. Storing both
# lets notebook 06 test whether the P2/P4 judge effects survive when the ORIGINAL rationale is not
# seeded with a gap-matched justification -- the key reviewer concern.
N_CASES = 50
GEN_CONDITIONS = ["gap_given", "gap_blind"]
src = PROC / "override_subset"
src = src if df_exists(src) else (PROC / "corpus_scored")
if not OPENAI_API_KEY:
    print("[skip] no OPENAI_API_KEY in .env")
elif not df_exists(src):
    print("[skip] run 01 and 02 first (no scored corpus found).")
else:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def llm_complete(model, system, user, max_tokens=700):
        for extra in (dict(temperature=0.2, max_tokens=max_tokens),
                      dict(max_completion_tokens=max_tokens), dict()):
            try:
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": user}], **extra)
                return r.choices[0].message.content
            except Exception as e:
                last = e
        raise last

    SYS = ("You are a credit rating analyst. Given a firm's financial profile, any context, and "
           "(when provided) the NOTCH_GAP between its final rating and its standalone financials, "
           "assign an issuer rating on the scale AAA (best) to D (default) and justify it. When a "
           "notch gap is given, devote at least one clear sentence to the qualitative/contextual "
           "factor (parent, group, or government support; industry; governance) that produces it. "
           "Return strict JSON: {\"rating\": <string>, \"rationale\": <string>}.")

    df = read_df(src).head(N_CASES).reset_index(drop=True)
    feat = json.loads((PROC / "feature_columns.json").read_text())
    text_col = next((c for c in df.columns
                     if c.lower() in ("context_text", "annual_report", "mdna", "text")), None)

    def gap_hint(row):
        d = row.get("delta")
        try:
            d = int(d)
        except Exception:
            d = 0
        if not d:
            return ""
        direction = "above" if d > 0 else "below"
        return (f"NOTCH_GAP: the final rating sits {abs(d)} notch(es) {direction} the standalone "
                f"financial profile because of an external qualitative factor; explain that factor "
                f"and its direction explicitly in the rationale.\n")

    rows = []
    for cond in GEN_CONDITIONS:
        for i, row in df.iterrows():
            profile = "; ".join(f"{c}={row[c]:.4g}" for c in feat if pd.notna(row[c]))
            context = str(row[text_col]) if text_col and pd.notna(row.get(text_col)) else ""
            try:
                delta = int(row.get("delta")) if pd.notna(row.get("delta")) else 0
            except Exception:
                delta = 0
            hint = gap_hint(row) if cond == "gap_given" else ""
            user = (f"Financial profile: {profile}\n"
                    + (f"Context: {context}\n" if context else "")
                    + hint + "Return JSON only.")
            for tag, model in LLM_MODELS.items():
                if not model:
                    continue
                try:
                    out = llm_complete(model, SYS, user)
                    try:
                        j = json.loads(out)
                        rating, rationale = j.get("rating", ""), j.get("rationale", "")
                    except Exception:
                        rating, rationale = "", out
                except Exception as e:
                    rating, rationale = "", f"ERROR: {type(e).__name__}: {e}"
                rows.append(dict(case_id=i, model_tag=tag, model=model, gen_condition=cond,
                                 fin_profile=profile, context=context, delta=delta,
                                 gen_rating=rating, gen_rationale=rationale))
    gen = pd.DataFrame(rows)
    write_df(gen, PROC / "generations")
    save_table(gen.groupby(["gen_condition", "model_tag"])["gen_rating"]
               .apply(lambda s: (s != "").mean()).round(3).rename("valid_rate").reset_index(),
               "03_generation_validrate")
    print("wrote generations:", len(gen), "rows | conditions:", GEN_CONDITIONS)


wrote generations: 300 rows | conditions: ['gap_given', 'gap_blind']
